In [22]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
from momentfm import MOMENTPipeline

from IEEE.prepare_IEEE import process_all_IEEE_files
from common.pipeline import build_moment_table
from common.classification import (train_test_split_LIAS, scale_features,
                                   train_svc_model, train_log_reg_model, evaluate_model)

In [24]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large",
    model_kwargs={'task_name': 'embedding'}, # We are loading the model in `embedding` mode
)
model.init()
model = model.to(device).float()

/home/mdereje/Projects/mcsa_tsfm/.venv/lib/python3.11/site-packages/momentfm/models/moment.py:174: UserWarning: Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.
  warnings.warn("Only reconstruction head is pre-trained. Classification and forecasting heads must be fine-tuned.")


In [ ]:
WINDOW_SIZE = 1024
OVERLAP = int(WINDOW_SIZE/2)
MAX_WINDOWS = 200  # windows kept per signal per recording (None keeps all: 1,641 per recording for a full export at 1024/512, which makes very large DataFrames)
batch_size = 256 # You can adjust this based on your GPU memory

# Windows -> MOMENT embeddings (per phase) -> rename_and_merge -> add_labels_IEEE, in one call (common/pipeline.py).
# add_labels_IEEE adds 'class' (from labels.csv) and 'file_group' (condition and load, e.g. 'r1b_torque05').
# Feature columns are named moment_<phase>_<number>, e.g. moment_a_17 (1024 * 3 = 3072 features).
df_labelled = build_moment_table("IEEE", model, device, WINDOW_SIZE, OVERLAP, batch_size=batch_size, max_windows=MAX_WINDOWS)
df_labelled.head()

In [ ]:
# ---- Split at file level while stratifying on file_group ---- #
# train_test_split_LIAS only needs 'file_name', 'window_num' and 'file_group', so it works on IEEE too.
# Change the seed to run the experiment on a different split.
train_df, test_df = train_test_split_LIAS(df_labelled, test_size=0.2, seed=67)

In [ ]:
# Checking this worked.  Should give an empty set
set(train_df["file_name"]) & set(test_df["file_name"])

In [ ]:
print(train_df["file_group"].value_counts(normalize=True))
print(test_df["file_group"].value_counts(normalize=True))

In [ ]:
# Standardized features (the scaler is fit on the training set only), for the models trained on scaled data
X_train_scaled, y_train, X_test_scaled, y_test = scale_features(train_df, test_df, feature_type="moment")

# Unscaled features, for the models trained on unscaled data
X_train = train_df.filter(regex="^moment_")
X_test = test_df.filter(regex="^moment_")

print(f"X_train_scaled: {X_train_scaled.shape}, X_test_scaled: {X_test_scaled.shape}")

In [ ]:
# Train SVM model with RBF kernel on UNscaled data
svm_model_unscaled_rbf = train_svc_model(X_train, y_train, kernel='rbf')

In [ ]:
# Make predictions on the unscaled test set
y_pred_unscaled_rbf = svm_model_unscaled_rbf.predict(X_test)

# Evaluate the model
evaluate_model(y_test, y_pred_unscaled_rbf, "RBF Kernel SVM on Unscaled Data")

In [ ]:
# Train SVM model with linear kernel on UNscaled data
svm_model_unscaled_linear = train_svc_model(X_train, y_train, kernel='linear')

In [ ]:
# Make predictions on the unscaled test set
y_pred_unscaled_linear = svm_model_unscaled_linear.predict(X_test)

# Evaluate the model
evaluate_model(y_test, y_pred_unscaled_linear, "Linear Kernel SVM on Unscaled Data")

In [ ]:
# Train SVM model with RBF kernel on scaled data
svm_model_scaled_rbf = train_svc_model(X_train_scaled, y_train, kernel='rbf')

In [ ]:
# Make predictions on the scaled test set
y_pred_scaled_rbf = svm_model_scaled_rbf.predict(X_test_scaled)

# Evaluate the model
evaluate_model(y_test, y_pred_scaled_rbf, "RBF Kernel SVM on Scaled Data")

In [ ]:
# Train SVM model with linear kernel on scaled data
svm_model_scaled_linear = train_svc_model(X_train_scaled, y_train, kernel='linear')

In [ ]:
# Make predictions on the scaled test set
y_pred_scaled_linear = svm_model_scaled_linear.predict(X_test_scaled)

# Evaluate the model
evaluate_model(y_test, y_pred_scaled_linear, "Linear Kernel SVM on Scaled Data")

In [ ]:
# Train Logistic Regression model on UNscaled data
log_reg_model_unscaled = train_log_reg_model(X_train, y_train)

# Make predictions on the unscaled test set
y_pred_log_reg_unscaled = log_reg_model_unscaled.predict(X_test)

# Evaluate the model
evaluate_model(y_test, y_pred_log_reg_unscaled, "Logistic Regression on Unscaled Data")

In [ ]:
# Train Logistic Regression model on SCALED data
log_reg_model_scaled = train_log_reg_model(X_train_scaled, y_train)

# Make predictions on the scaled test set
y_pred_log_reg_scaled = log_reg_model_scaled.predict(X_test_scaled)

# Evaluate the model
evaluate_model(y_test, y_pred_log_reg_scaled, "Logistic Regression on Scaled Data")

## Raw windows (exploration)

In [ ]:
all_processed_windows_a, all_processed_windows_b, all_processed_windows_c = process_all_IEEE_files(WINDOW_SIZE, OVERLAP, MAX_WINDOWS)

In [ ]:
def windows_to_dataframe(processed_windows):
    rows = []

    for window in processed_windows:
        row = window["metadata"].copy()

        for i, value in enumerate(window["data"]):
            row[f"sample_{i}"] = value

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
# Step 2: Convert windows to DataFrames
df_a = windows_to_dataframe(all_processed_windows_a)
df_b = windows_to_dataframe(all_processed_windows_b)
df_c = windows_to_dataframe(all_processed_windows_c)

# To get time we just have to divide the sample number by the sampling frequency (50 kHz here)
for df in (df_a, df_b, df_c):
    parts = df["file_name"].str.extract(r"^(?P<condition>\w+?)_torque(?P<torque>\d+)_(?P<repetition>\d+)")
    loc = df.columns.get_loc("sample_0")
    df.insert(loc, "condition", parts["condition"])
    df.insert(loc + 1, "torque", parts["torque"].astype(int) / 10)
    df.insert(loc + 1, "repetition", parts["repetition"].astype(int)) 



In [ ]:
df_a